# Diffusion 실습 — MNIST로 DDPM 처음부터 학습하기

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/diffusion_ddpm_mnist.ipynb)

**먼저:** 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 를 고른 뒤 위에서부터 차례로 실행하세요.

1. 앞방향(forward): 깨끗한 이미지에 노이즈를 조금씩 섞어 완전한 노이즈로 만든다.
2. 학습: 작은 UNet이 "섞인 노이즈 $\epsilon$"을 맞히도록 학습한다. 손실 $\|\epsilon - \epsilon_\theta(x_t, t)\|^2$
3. 뒷방향(reverse): 순수 노이즈에서 시작해 예측한 노이즈를 조금씩 빼면 숫자가 나타난다.

T4 GPU 기준 학습은 약 10분이 걸립니다.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, math, copy
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print('device:', device)

## 1. 데이터: MNIST를 [-1, 1] 범위로

In [ ]:
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
ds = datasets.MNIST('data', train=True, download=True, transform=tf)
loader = DataLoader(ds, batch_size=128, shuffle=True, num_workers=2, drop_last=True)
x0, _ = next(iter(loader))
print(x0.shape, x0.min().item(), x0.max().item())

## 2. 노이즈 스케줄과 앞방향 과정

$\beta_t$ 를 1e-4 에서 0.02 까지 선형으로 늘리고 $\bar\alpha_t = \prod_{s\le t}(1-\beta_s)$ 라 두면, 한 번에

$$x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon,\quad \epsilon\sim\mathrm{N}(0, I)$$

로 아무 단계 $t$ 의 노이즈 섞인 이미지를 바로 만들 수 있습니다.

In [ ]:
T = 1000
betas = torch.linspace(1e-4, 0.02, T, device=device)
alphas = 1 - betas
alpha_bar = torch.cumprod(alphas, dim=0)

def q_sample(x0, t, noise):
    ab = alpha_bar[t].view(-1, 1, 1, 1)
    return ab.sqrt() * x0 + (1 - ab).sqrt() * noise

# 한 장에 노이즈가 섞여 가는 모습
img = x0[:1].to(device)
steps = [0, 50, 100, 200, 400, 600, 800, 999]
fig, axes = plt.subplots(1, len(steps), figsize=(14, 2))
for ax, t in zip(axes, steps):
    xt = q_sample(img, torch.tensor([t], device=device), torch.randn_like(img))
    ax.imshow(xt[0, 0].cpu(), cmap='gray', vmin=-1, vmax=1); ax.set_title(f't={t}'); ax.axis('off')
plt.show()

## 3. 노이즈 예측 모델: 작은 UNet

입력은 노이즈 섞인 이미지 $x_t$ 와 단계 $t$ 이고, 출력은 같은 크기의 노이즈 예측 $\epsilon_\theta(x_t, t)$ 입니다.
$t$ 는 Transformer 위치 인코딩과 같은 sin/cos 임베딩으로 넣습니다.
`num_classes` 와 `y` 는 6절(숫자를 지정해서 생성하기)에서 쓰므로 지금은 무시해도 됩니다.

In [ ]:
def time_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
    args = t.float()[:, None] * freqs[None]
    return torch.cat([args.sin(), args.cos()], dim=1)

class Block(nn.Module):
    def __init__(self, cin, cout, tdim):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, padding=1)
        self.conv2 = nn.Conv2d(cout, cout, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, cout)
        self.norm2 = nn.GroupNorm(8, cout)
        self.tproj = nn.Linear(tdim, cout)
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()
    def forward(self, x, temb):
        h = F.silu(self.norm1(self.conv1(x)))
        h = h + self.tproj(temb)[:, :, None, None]
        h = F.silu(self.norm2(self.conv2(h)))
        return h + self.skip(x)

class UNet(nn.Module):
    def __init__(self, ch=64, tdim=128, num_classes=None):
        super().__init__()
        self.tdim = tdim
        self.tmlp = nn.Sequential(nn.Linear(tdim, tdim), nn.SiLU(), nn.Linear(tdim, tdim))
        # 숫자 레이블 임베딩 (= 원핫 x 선형층). 마지막 칸(num_classes)은 '조건 없음'
        self.label_emb = nn.Embedding(num_classes + 1, tdim) if num_classes else None
        self.inc = nn.Conv2d(1, ch, 3, padding=1)
        self.down1 = Block(ch, ch, tdim)          # 28x28
        self.down2 = Block(ch, ch * 2, tdim)      # 14x14
        self.mid = Block(ch * 2, ch * 2, tdim)    # 7x7
        self.up2 = Block(ch * 4, ch, tdim)        # 14x14
        self.up1 = Block(ch * 2, ch, tdim)        # 28x28
        self.out = nn.Conv2d(ch, 1, 1)
    def forward(self, x, t, y=None):
        temb = self.tmlp(time_embedding(t, self.tdim))
        if self.label_emb is not None:
            temb = temb + self.label_emb(y)
        h1 = self.down1(self.inc(x), temb)
        h2 = self.down2(F.avg_pool2d(h1, 2), temb)
        m = self.mid(F.avg_pool2d(h2, 2), temb)
        u2 = self.up2(torch.cat([F.interpolate(m, scale_factor=2), h2], 1), temb)
        u1 = self.up1(torch.cat([F.interpolate(u2, scale_factor=2), h1], 1), temb)
        return self.out(u1)

model = UNet().to(device)
print(f'parameters: {sum(p.numel() for p in model.parameters()):,}')

## 4. 학습: 섞인 노이즈 맞히기

배치마다 단계 $t$ 를 무작위로 뽑고, 노이즈를 섞은 뒤, 모델이 그 노이즈를 맞히도록 MSE로 학습합니다.

**EMA**: 학습 중인 가중치를 그대로 쓰지 않고, 가중치의 지수이동평균(`ema`)을 따로 들고 있다가 생성할 때 씁니다.
매 스텝 흔들리는 가중치를 평균 내 주기 때문에 diffusion 모델에서는 생성 품질이 눈에 띄게 좋아집니다.

In [ ]:
EPOCHS = 20   # T4 기준 한 에폭에 30초 정도 걸립니다.
EMA_DECAY = 0.999

def train(model, use_labels=False, p_uncond=0.1):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, total_steps=EPOCHS * len(loader))
    ema = copy.deepcopy(model).eval().requires_grad_(False)
    losses = []
    for epoch in range(EPOCHS):
        for x0, y in loader:
            x0, y = x0.to(device), y.to(device)
            t = torch.randint(0, T, (x0.size(0),), device=device)
            noise = torch.randn_like(x0)
            if use_labels:   # 6절: p_uncond 확률로 레이블을 '조건 없음'(10)으로 바꿔 넣는다
                y = torch.where(torch.rand(y.shape, device=device) < p_uncond, 10, y)
                pred = model(q_sample(x0, t, noise), t, y)
            else:
                pred = model(q_sample(x0, t, noise), t)
            loss = F.mse_loss(pred, noise)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            with torch.no_grad():
                for pe, pm in zip(ema.parameters(), model.parameters()):
                    pe.lerp_(pm, 1 - EMA_DECAY)
            losses.append(loss.item())
        print(f'epoch {epoch + 1}/{EPOCHS}  loss {sum(losses[-100:]) / 100:.4f}')
    return ema, losses

ema, losses = train(model)

plt.plot(losses); plt.xlabel('step'); plt.ylabel('MSE loss'); plt.yscale('log'); plt.show()

## 5. 생성: 순수 노이즈에서 숫자로

$x_T\sim\mathrm{N}(0,I)$ 에서 시작해 $t=T-1,\dots,0$ 순서로

1. 예측한 노이즈로 원본을 추정합니다: $\hat x_0 = \big(x_t - \sqrt{1-\bar\alpha_t}\,\epsilon_\theta(x_t,t)\big)/\sqrt{\bar\alpha_t}$, 그리고 $[-1,1]$ 로 자릅니다.
2. $\hat x_0$ 와 $x_t$ 를 섞어 한 단계 덜 시끄러운 $x_{t-1}$ 을 만듭니다:

$$x_{t-1} = \frac{\sqrt{\bar\alpha_{t-1}}\,\beta_t}{1-\bar\alpha_t}\,\hat x_0 + \frac{\sqrt{\alpha_t}\,(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}\,x_t + \sigma_t z,\qquad \sigma_t^2 = \frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}\beta_t$$

(마지막 단계에서는 $z=0$). 1번에서 자르지 않아도 수식은 같지만, 자르면 초반의 큰 오차가 쌓이지 않아 결과가 깔끔해집니다.

In [ ]:
@torch.no_grad()
def sample(eps_fn, n=16, keep=(999, 700, 400, 200, 100, 50, 0)):
    x = torch.randn(n, 1, 28, 28, device=device)
    snapshots = {}
    for t in reversed(range(T)):
        tt = torch.full((n,), t, device=device)
        eps = eps_fn(x, tt)
        ab, ab_prev = alpha_bar[t], (alpha_bar[t - 1] if t > 0 else torch.tensor(1.0, device=device))
        x0_hat = ((x - (1 - ab).sqrt() * eps) / ab.sqrt()).clamp(-1, 1)
        x = (ab_prev.sqrt() * betas[t] * x0_hat + alphas[t].sqrt() * (1 - ab_prev) * x) / (1 - ab)
        if t > 0:
            x = x + ((1 - ab_prev) / (1 - ab) * betas[t]).sqrt() * torch.randn_like(x)
        if t in keep:
            snapshots[t] = x.clamp(-1, 1).cpu()
    return x.clamp(-1, 1).cpu(), snapshots

samples, snaps = sample(lambda x, t: ema(x, t), n=32)

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for ax, img in zip(axes.flat, samples):
    ax.imshow(img[0], cmap='gray'); ax.axis('off')
plt.suptitle('generated samples'); plt.show()

In [ ]:
# 노이즈가 숫자로 바뀌어 가는 과정 (위에서 만든 첫 번째 샘플)
ts = sorted(snaps, reverse=True)
fig, axes = plt.subplots(1, len(ts), figsize=(14, 2))
for ax, t in zip(axes, ts):
    ax.imshow(snaps[t][0, 0], cmap='gray'); ax.set_title(f't={t}'); ax.axis('off')
plt.show()

## 6. 숫자를 지정해서 생성하기: 조건부 생성 + CFG

지금까지의 모델은 숫자를 **무작위로** 그립니다. "7을 그려"라고 지정하려면 단계 $t$ 를 넣었던 것처럼 **숫자 레이블 $y$ 도 넣어 주면** 됩니다.
3절 `UNet(num_classes=10)` 이 레이블 임베딩(원핫 x 선형층과 같음)을 시간 임베딩에 더해 줍니다.

레이블만 넣으면 가끔 "7인 듯 1인 듯" 애매한 숫자가 나옵니다. 그래서 **Classifier-Free Guidance(CFG)** 를 함께 씁니다.

- **학습:** 손실은 그대로입니다. 10% 확률로 레이블을 '조건 없음'(10번)으로 바꿔 넣어서, 모델 하나가 조건 있는 예측과 없는 예측을 둘 다 배우게 합니다.
- **생성:** 매 단계 모델을 두 번 부르고 그 차이를 키웁니다.

$$\epsilon = \epsilon_\text{uncond} + w\,(\epsilon_y - \epsilon_\text{uncond})$$

$\epsilon_y - \epsilon_\text{uncond}$ 이 "숫자 $y$ 다움" 방향이고, $w$ 는 그 방향을 얼마나 세게 밀지 정합니다. Stable Diffusion의 `guidance_scale` 이 바로 이 $w$ 입니다.

학습을 한 번 더 하므로 T4 기준 10분 정도 걸립니다.

In [ ]:
cmodel = UNet(num_classes=10).to(device)
cema, closses = train(cmodel, use_labels=True)

In [ ]:
NULL = 10   # '조건 없음' 레이블

def cfg_eps(y, w):
    def eps_fn(x, t):
        # 조건 있음/없음을 한 배치로 묶어 모델을 한 번만 부른다
        e_cond, e_uncond = cema(torch.cat([x, x]), torch.cat([t, t]),
                                torch.cat([y, torch.full_like(y, NULL)])).chunk(2)
        return e_uncond + w * (e_cond - e_uncond)
    return eps_fn

# 0~9 를 한 줄씩 8개씩 (w = 3)
y = torch.arange(10, device=device).repeat_interleave(8)
imgs, _ = sample(cfg_eps(y, w=3.0), n=len(y))

fig, axes = plt.subplots(10, 8, figsize=(8, 10))
for ax, img in zip(axes.flat, imgs):
    ax.imshow(img[0], cmap='gray'); ax.axis('off')
plt.suptitle('class-conditional samples (row = digit 0-9, w = 3)'); plt.show()

In [ ]:
# CFG 세기 w 비교: 같은 초기 노이즈, 숫자 0~9
ws = [0.0, 1.0, 3.0, 7.0]
y = torch.arange(10, device=device)
fig, axes = plt.subplots(len(ws), 10, figsize=(10, 4.5))
for row, w in zip(axes, ws):
    torch.manual_seed(0)
    imgs, _ = sample(cfg_eps(y, w), n=10)
    for ax, img in zip(row, imgs):
        ax.imshow(img[0], cmap='gray'); ax.axis('off')
    row[0].set_title(f'w = {w:g}', loc='left', fontsize=10)
plt.tight_layout(); plt.show()

- $w=0$: 조건을 무시하므로 레이블과 상관없는 숫자가 나옵니다.
- $w=1$: 레이블을 그냥 넣은 것과 같습니다.
- $w$ 를 키울수록 숫자가 또렷해지지만, 너무 크면 획이 굵어지고 모양이 비슷비슷해집니다.

## 생각해 볼 것

- `EPOCHS` 를 2로 줄이거나, 생성할 때 `ema` 대신 `model` 을 쓰면 생성 품질이 어떻게 달라지나요?
- `p_uncond` 를 0으로 두고 학습하면 CFG가 왜 동작하지 않을까요?
- 생성할 때 `T` 단계를 모두 밟지 않고 건너뛰면 어떻게 될까요? (힌트: DDIM)
- Stable Diffusion은 같은 원리를 **이미지 대신 VAE의 잠재 공간**에서, 숫자 레이블 대신 **텍스트 임베딩**을 조건으로 붙여(CFG 포함) 돌린 것입니다.

## (선택) 같은 원리의 대형 모델: Stable Diffusion 1.5

가중치를 약 4GB 내려받으므로 처음 실행은 몇 분 걸립니다. T4 GPU에서 한 장에 몇 초 정도입니다.

In [ ]:
!pip -q install diffusers transformers accelerate

from diffusers import StableDiffusionPipeline
pipe = StableDiffusionPipeline.from_pretrained(
    'stable-diffusion-v1-5/stable-diffusion-v1-5', torch_dtype=torch.float16).to('cuda')

image = pipe('a watercolor painting of a cat reading a book', num_inference_steps=25, guidance_scale=7.5).images[0]
image